# Comparacao de Algoritmos de Ordenacao

Este notebook compara **tempo de execucao** e **numero de comparacoes** de:

- Bubble Sort
- Selection Sort
- Selection Sort Alternativo (interrompe na primeira rodada se a lista ja estiver ordenada)

In [ ]:
from time import perf_counter
import random
import pandas as pd
import matplotlib.pyplot as plt

random.seed(42)

In [ ]:
def bubble_sort_conta(lista):
    a = lista.copy()
    comparacoes = 0
    n = len(a)

    for i in range(n - 1):
        trocou = False
        for j in range(n - 1 - i):
            comparacoes += 1
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                trocou = True
        if not trocou:
            break

    return a, comparacoes


def selection_sort_conta(lista):
    a = lista.copy()
    comparacoes = 0
    n = len(a)

    for i in range(n - 1):
        min_idx = i
        for j in range(i + 1, n):
            comparacoes += 1
            if a[j] < a[min_idx]:
                min_idx = j
        if min_idx != i:
            a[i], a[min_idx] = a[min_idx], a[i]

    return a, comparacoes


def selection_sort_alternativo_conta(lista):
    a = lista.copy()
    comparacoes = 0
    n = len(a)

    # Primeira rodada: se nenhuma troca seria necessaria e nao houver quebra de ordem,
    # consideramos a lista ordenada e encerramos cedo.
    min_idx = 0
    tem_quebra = False
    for j in range(1, n):
        comparacoes += 1
        if a[j] < a[min_idx]:
            min_idx = j

        comparacoes += 1
        if a[j] < a[j - 1]:
            tem_quebra = True

    if n <= 1:
        return a, comparacoes

    if (min_idx == 0) and (not tem_quebra):
        return a, comparacoes

    if min_idx != 0:
        a[0], a[min_idx] = a[min_idx], a[0]

    for i in range(1, n - 1):
        min_idx = i
        for j in range(i + 1, n):
            comparacoes += 1
            if a[j] < a[min_idx]:
                min_idx = j
        if min_idx != i:
            a[i], a[min_idx] = a[min_idx], a[i]

    return a, comparacoes

In [ ]:
def medir_algoritmo(func, lista, repeticoes=5):
    tempos = []
    comparacoes = None

    for _ in range(repeticoes):
        inicio = perf_counter()
        ordenada, comps = func(lista)
        fim = perf_counter()

        assert ordenada == sorted(lista), f"Erro de ordenacao em {func.__name__}"
        tempos.append(fim - inicio)
        comparacoes = comps

    return sum(tempos) / len(tempos), comparacoes


def gerar_casos(n):
    return {
        "ordenada": list(range(n)),
        "quase_ordenada": list(range(n - 1)) + [n - 2],
        "aleatoria": random.sample(range(10 * n), n),
        "reversa": list(range(n, 0, -1)),
    }

In [ ]:
# Ajuste os tamanhos conforme sua necessidade.
tamanhos = [100, 300, 700]
repeticoes = 7

algoritmos = {
    "Bubble Sort": bubble_sort_conta,
    "Selection Sort": selection_sort_conta,
    "Selection Sort Alt": selection_sort_alternativo_conta,
}

resultados = []

for n in tamanhos:
    casos = gerar_casos(n)
    for nome_caso, lista in casos.items():
        for nome_algo, func in algoritmos.items():
            tempo_medio, comps = medir_algoritmo(func, lista, repeticoes=repeticoes)
            resultados.append({
                "n": n,
                "caso": nome_caso,
                "algoritmo": nome_algo,
                "tempo_medio_s": tempo_medio,
                "comparacoes": comps,
            })

df = pd.DataFrame(resultados)
df.sort_values(["n", "caso", "tempo_medio_s"], inplace=True)
df

In [ ]:
# Tabela resumida por caso e tamanho
resumo = df.pivot_table(
    index=["n", "caso"],
    columns="algoritmo",
    values=["tempo_medio_s", "comparacoes"],
    aggfunc="first"
)
resumo

In [ ]:
# Graficos
for metrica in ["tempo_medio_s", "comparacoes"]:
    for caso in df["caso"].unique():
        fig, ax = plt.subplots(figsize=(8, 4.5))
        parte = df[df["caso"] == caso]

        for nome_algo in parte["algoritmo"].unique():
            dados_algo = parte[parte["algoritmo"] == nome_algo]
            ax.plot(dados_algo["n"], dados_algo[metrica], marker="o", label=nome_algo)

        ax.set_title(f"{metrica} | caso: {caso}")
        ax.set_xlabel("Tamanho da lista (n)")
        ax.set_ylabel(metrica)
        ax.grid(alpha=0.3)
        ax.legend()
        plt.show()